# Volume da aorta e qualidade visual da segmentação

## Objetivo

Analisa a quantidade de voxels da máscara final da aorta, sua ocupação percentual no volume processado e a área média por fatia. A análise usa rótulos visuais independentes para os conjuntos de treino e validação, relacionando a qualidade da aorta com a localização dos óstios e o Dice arterial.

O corte exploratório é estimado somente no treino e aplicado sem recalibração na validação.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Localiza a raiz antes dos imports internos a partir de qualquer subpasta de EDA.
current = Path.cwd().resolve()
REPO_ROOT = next(
    path
    for path in [current, *current.parents]
    if (path / "src").exists() and (path / "output").exists()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    get_aorta_visual_review,
    load_aorta_review_cohort,
    load_aorta_visual_reviews,
)
from utils.project.runtime.notebook_env import (
    configure_notebook_environment,  # noqa: E402
)

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## Configuração

Os rótulos visuais abaixo foram definidos após inspeção dos HTMLs 3D de cada run. `both correct` e `both tolerable` são considerados sucesso dos óstios.

Treino e validação usam as referências atuais com limite inferior fixo em -300 HU e limite superior P99.9. As classificações e observações da inspeção visual são carregadas de `config/aorta_visual_reviews.json`.

In [ ]:
REVIEW_CONFIG_PATH = REPO_ROOT / "config/aorta_visual_reviews.json"
REVIEW_CATALOG = load_aorta_visual_reviews(REVIEW_CONFIG_PATH)
TRAIN_REVIEW = get_aorta_visual_review(REVIEW_CATALOG, "normal", "train")
VAL_REVIEW = get_aorta_visual_review(REVIEW_CATALOG, "normal", "val")
TRAIN_RUN_DIR = REPO_ROOT / TRAIN_REVIEW["run_dir"]
VAL_RUN_DIR = REPO_ROOT / VAL_REVIEW["run_dir"]

print(
    f"Treino: {len(TRAIN_REVIEW['aorta_good_ids'])} boas e "
    f"{len(TRAIN_REVIEW['aorta_bad_ids'])} ruins"
)
print(
    f"Validação: {len(VAL_REVIEW['aorta_good_ids'])} boas e "
    f"{len(VAL_REVIEW['aorta_bad_ids'])} ruins"
)

## Carregamento e métricas derivadas

In [ ]:
REQUIRED_COLUMNS = {
    "IMG_ID",
    "artery_dice",
    "ostia_detection_status",
    "left_ostium_distance_mm",
    "right_ostium_distance_mm",
    "image_slice_count",
    "aorta_circle_count",
    "aorta_mask_voxel_count",
    "aorta_segmented_slice_count",
    "aorta_voxels_per_segmented_slice",
    "aorta_volume_fraction",
}

train_df = load_aorta_review_cohort(
    REPO_ROOT,
    TRAIN_REVIEW,
    "train",
    cohort_name="treino",
    required_columns=REQUIRED_COLUMNS,
)
val_df = load_aorta_review_cohort(
    REPO_ROOT,
    VAL_REVIEW,
    "val",
    cohort_name="validação",
    required_columns=REQUIRED_COLUMNS,
)

# A maior distância resume o pior resultado entre os dois óstios de cada exame.
for cohort_df in (train_df, val_df):
    cohort_df["max_ostium_distance_mm"] = cohort_df[
        ["left_ostium_distance_mm", "right_ostium_distance_mm"]
    ].max(axis=1)

# Mantém o nome curto usado na primeira parte do notebook.
df = train_df
print(f"Treino carregado: {len(train_df)} imagens")
print(f"Validação carregada: {len(val_df)} imagens")

## Funções de análise

Os helpers abaixo garantem que treino e validação usem exatamente as mesmas agregações e configurações visuais.

In [ ]:
from eda.aorta.view.aorta_volume_quality_views import (
    plot_ordered_volume,
    plot_ostia_relationship,
    plot_visual_quality,
)
from utils.project.analysis.aorta_volume_quality import (
    build_ostia_summary,
    build_overview_row,
    build_quality_crosstab,
    build_visual_summary,
    evaluate_volume_threshold,
    find_volume_threshold,
)

## 1. Visão geral das coortes

Treino e validação aparecem na mesma tabela, uma linha por coorte, pois estas métricas não dependem de agrupamentos internos.

In [ ]:
overview = pd.DataFrame(
    [
        build_overview_row(train_df, "treino"),
        build_overview_row(val_df, "validação"),
    ]
)
display(overview.round(3))

## 2. Métricas da aorta em relação ao sucesso dos óstios

Treino e validação utilizam a mesma agregação, facilitando a comparação direta dos volumes, número de fatias, erro dos óstios e Dice.

### 2.1. Resumo numérico por desfecho dos óstios

In [ ]:
print("Treino")
train_ostia_summary = build_ostia_summary(train_df)
display(train_ostia_summary.round(3))

In [ ]:
print("Validação")
val_ostia_summary = build_ostia_summary(val_df)
display(val_ostia_summary.round(3))

### 2.2. Relação entre volume da aorta e erro dos óstios

In [ ]:
plot_ostia_relationship(train_df, "treino")

In [ ]:
plot_ostia_relationship(val_df, "validação")

## 3. Qualidade visual da aorta

Cada subseção responde a uma pergunta específica e apresenta treino e validação em células consecutivas.

### 3.1. Qualidade visual versus desfecho dos óstios

In [ ]:
print("Treino")
train_quality_vs_ostia = build_quality_crosstab(train_df)
display(train_quality_vs_ostia)

In [ ]:
print("Validação")
val_quality_vs_ostia = build_quality_crosstab(val_df)
display(val_quality_vs_ostia)

### 3.2. Resumo por qualidade visual

In [ ]:
print("Treino")
train_visual_summary = build_visual_summary(train_df)
display(train_visual_summary.round(3))

In [ ]:
print("Validação")
val_visual_summary = build_visual_summary(val_df)
display(val_visual_summary.round(3))

### 3.3. Distribuição do volume e da área por fatia

In [ ]:
print("Treino")
plot_visual_quality(train_df, "treino")

In [ ]:
print("Validação")
plot_visual_quality(val_df, "validação")

### 3.4. Ocupação percentual por exame

In [ ]:
print("Treino")
plot_ordered_volume(train_df, "treino")

In [ ]:
print("Validação")
plot_ordered_volume(val_df, "validação")

## 4. Faixa exploratória para triagem visual

O corte é ajustado somente no treino e aplicado sem recalibração na validação.

### 4.1. Desempenho do corte no treino e na validação

In [ ]:
print("Treino")
train_threshold_df = find_volume_threshold(train_df)
train_best_threshold = float(train_threshold_df.iloc[0]["threshold_percentual"])
display(train_threshold_df.head(5).round(4))
print(f"Corte selecionado no treino: {train_best_threshold:.4f}%")

In [ ]:
print("Validação")
val_threshold_evaluation = evaluate_volume_threshold(
    val_df,
    train_best_threshold,
)
display(val_threshold_evaluation.round(4))

## 5. Síntese comparativa

A tabela reúne os indicadores necessários para comparar aortas boas e ruins entre as duas coortes.

In [ ]:
cohort_comparison = pd.concat(
    [
        train_df.assign(coorte="treino"),
        val_df.assign(coorte="validação"),
    ],
    ignore_index=True,
)
cohort_summary = (
    cohort_comparison.groupby(["coorte", "visual_aorta_quality"], observed=True)
    .agg(
        imagens=("IMG_ID", "size"),
        sucesso_ostios=("ostia_success", "mean"),
        volume_percentual_medio=("aorta_volume_percentage", "mean"),
        volume_percentual_mediano=("aorta_volume_percentage", "median"),
        voxels_por_fatia_media=("aorta_voxels_per_segmented_slice", "mean"),
        dice_arterial_medio=("artery_dice", "mean"),
    )
    .reset_index()
)
display(cohort_summary.round(4))

## Conclusão

A organização por métrica permite comparar treino e validação imediatamente. A seção redundante de aortas confirmadas foi removida, pois seus indicadores já estão disponíveis no resumo por qualidade visual e no cruzamento com o desfecho dos óstios.

In [ ]:
train_good = train_df.loc[train_df["visual_aorta_quality"].eq("boa")]
train_bad = train_df.loc[train_df["visual_aorta_quality"].eq("ruim")]
val_good = val_df.loc[val_df["visual_aorta_quality"].eq("boa")]
val_bad = val_df.loc[val_df["visual_aorta_quality"].eq("ruim")]

print("Síntese da análise:")
print(
    f"- Treino: {len(train_good)}/{len(train_df)} aortas boas "
    f"({len(train_good) / len(train_df):.1%})."
)
print(
    f"- Validação: {len(val_good)}/{len(val_df)} aortas boas "
    f"({len(val_good) / len(val_df):.1%})."
)
print(
    f"- Volume médio treino — boas: {train_good['aorta_volume_percentage'].mean():.2f}%; "
    f"ruins: {train_bad['aorta_volume_percentage'].mean():.2f}%."
)
print(
    f"- Volume médio validação — boas: {val_good['aorta_volume_percentage'].mean():.2f}%; "
    f"ruins: {val_bad['aorta_volume_percentage'].mean():.2f}%."
)
print(
    f"- Corte aprendido no treino: {train_best_threshold:.2f}%; "
    f"acurácia balanceada na validação: "
    f"{val_threshold_evaluation.iloc[0]['acuracia_balanceada']:.1%}."
)
print(
    f"- Sucesso dos óstios nas aortas boas da validação: "
    f"{val_good['ostia_success'].mean():.1%}."
)
print(
    "- O exame 821 foi mantido como bom, com ressalva de que poderia ser segmentado um pouco mais."
)